# Validation Complète YOLO (OBB / Segmentation / Détection) — Maximum de Métriques

Ce notebook exécute la validation complète (`model.val(...)`) sur votre modèle entraîné (`best.pt`) et extrait **le maximum de métriques et graphiques** :
- **Bloc 1** : Configuration des chemins (`MODEL_PATH`, `DATA_YAML`, split `val` ou `test`, paramètres d'évaluation).
- **Bloc 2** : Exécution de `model.val(...)` avec génération des courbes et matrices (`plots=True`, `save_json=True`).
- **Bloc 3** : Extraction détaillée de toutes les métriques globales et par classe (Précision, Rappel, F1-score, `mAP50`, `mAP75`, `mAP50-95`, fitness, temps d'inférence, matrice de confusion, métriques par image).
- **Bloc 4** : Affichage des courbes générées (Matrice de confusion, PR-curve, F1-curve, P-curve, R-curve) et export en CSV / JSON / Excel.

## Bloc 1 — Configuration des chemins et paramètres de validation

In [19]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
from ultralytics import YOLO

# 1. Chemin vers votre modèle entraîné (ex: Module 3 OBB ou Module 1 Seg)
MODEL_PATH = Path(r"D:\S5\Projet_d_expertise\projet\models\03_orientation_des_composants\weights\best.pt").resolve()

# 2. Chemin vers le fichier data.yaml correspondant
DATA_YAML = Path(r"D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml").resolve()

# 3. Split à évaluer ('val' ou 'test')
EVAL_SPLIT = "val"

# 4. Paramètres de validation
IMG_SIZE = 640
BATCH_SIZE = 16
IOU_THRESHOLD = 0.7
# Laisser CONF_THRESHOLD = None pour le calcul standard mAP (recommandé), ou fixer ex. 0.25 pour matrice opérationnelle
CONF_THRESHOLD = None

OUTPUT_DIR = Path(r"D:\S5\Projet_d_expertise\projet\evaluations").resolve()
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Configuration de validation :")
print("  - Modèle   :", MODEL_PATH)
print("  - Dataset  :", DATA_YAML)
print("  - Split    :", EVAL_SPLIT)
print("  - Device   :", DEVICE)

Configuration de validation :
  - Modèle   : D:\S5\Projet_d_expertise\projet\models\03_orientation_des_composants\weights\best.pt
  - Dataset  : D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml
  - Split    : val
  - Device   : cpu


## Bloc 2 — Lancement de la validation complète (`model.val`)

In [20]:
# Recherche automatique du dernier best.pt si le chemin par défaut n'existe pas
if not MODEL_PATH.exists():
    found_models = sorted(
        Path(r"D:\S5\Projet_d_expertise\projet\models").glob("**/best.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if found_models:
        MODEL_PATH = found_models[0].resolve()
        print(f"Modèle trouvé automatiquement : {MODEL_PATH}")
    else:
        raise FileNotFoundError(f"Modèle introuvable : {MODEL_PATH}")

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Fichier data.yaml introuvable : {DATA_YAML}")

model = YOLO(str(MODEL_PATH))
print(f"Tâche du modèle : {model.task} | Classes : {model.names}")

val_kwargs = dict(
    data=str(DATA_YAML),
    split=EVAL_SPLIT,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    iou=IOU_THRESHOLD,
    device=DEVICE,
    plots=True,
    save_json=True,
    project=str(OUTPUT_DIR),
    name=f"{MODEL_PATH.parent.parent.name}_{EVAL_SPLIT}",
    exist_ok=True,
)
if CONF_THRESHOLD is not None:
    val_kwargs["conf"] = CONF_THRESHOLD

metrics = model.val(**val_kwargs)
save_dir = Path(metrics.save_dir)
print(f"Validation terminée. Résultats sauvegardés dans : {save_dir}")

Tâche du modèle : obb | Classes : {0: 'cable', 1: 'cable-tie', 2: 'clip', 3: 'connector', 4: 'label'}
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
YOLO11n-obb summary (fused): 110 layers, 2,654,698 parameters, 0 gradients, 6.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 91.841.2 MB/s, size: 34.8 KB)
val: Scanning D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\valid\labels.cache... 22 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 22/22  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.5s/it 7.1s<13.1s
                   all         22        106      0.955      0.356      0.531      0.399
                 cable          3         31          1          0     0.0553     0.0139
             cable-tie          2          2          1          0      0.745      0.548
                  clip          1          5       

## Bloc 3 — Extraction de toutes les métriques (Globales, par classe, par image, vitesse & confusion)

In [21]:
# 1. Dictionnaire brut de toutes les métriques scalaires Ultralytics
results_dict = getattr(metrics, "results_dict", {})
df_global = pd.DataFrame([
    {"Metrique": k, "Valeur": round(float(v), 6)} for k, v in results_dict.items()
])

# 2. Extraction détaillée du bloc principal (obb, seg ou box)
task_metric_obj = None
for attr in ["obb", "seg", "box"]:
    candidate = getattr(metrics, attr, None)
    if candidate is not None and hasattr(candidate, "map"):
        task_metric_obj = candidate
        task_label = attr.upper()
        break

extra_rows = []
if task_metric_obj is not None:
    p_mean = float(getattr(task_metric_obj, "mp", np.nan))
    r_mean = float(getattr(task_metric_obj, "mr", np.nan))
    f1_mean = (2 * p_mean * r_mean / (p_mean + r_mean)) if (p_mean + r_mean) > 0 else 0.0
    extra_rows.extend([
        {"Metrique": f"{task_label}/Precision_moyenne (mp)", "Valeur": round(p_mean, 6)},
        {"Metrique": f"{task_label}/Recall_moyen (mr)", "Valeur": round(r_mean, 6)},
        {"Metrique": f"{task_label}/F1_moyen", "Valeur": round(f1_mean, 6)},
        {"Metrique": f"{task_label}/mAP50", "Valeur": round(float(getattr(task_metric_obj, "map50", np.nan)), 6)},
        {"Metrique": f"{task_label}/mAP75", "Valeur": round(float(getattr(task_metric_obj, "map75", np.nan)), 6)},
        {"Metrique": f"{task_label}/mAP50-95", "Valeur": round(float(getattr(task_metric_obj, "map", np.nan)), 6)},
    ])

# 3. Vitesse d'exécution (ms / image)
speed = getattr(metrics, "speed", {})
for sp_k, sp_v in speed.items():
    extra_rows.append({"Metrique": f"Speed_{sp_k}_ms", "Valeur": round(float(sp_v), 3)})
if speed:
    total_ms = sum(float(v) for v in speed.values())
    extra_rows.append({"Metrique": "Speed_total_ms", "Valeur": round(total_ms, 3)})
    if total_ms > 0:
        extra_rows.append({"Metrique": "FPS_estime", "Valeur": round(1000.0 / total_ms, 2)})

df_global = pd.concat([df_global, pd.DataFrame(extra_rows)], ignore_index=True).drop_duplicates(subset=["Metrique"])
print("=== MÉTRIQUES GLOBALES ===")
display(df_global)

# 4. Métriques détaillées PAR CLASSE
per_class_rows = []
if task_metric_obj is not None and hasattr(task_metric_obj, "ap_class_index"):
    class_indices = task_metric_obj.ap_class_index
    for idx_pos, cid in enumerate(class_indices):
        cid_int = int(cid)
        cname = model.names.get(cid_int, str(cid_int))
        p_c = float(task_metric_obj.p[idx_pos]) if idx_pos < len(task_metric_obj.p) else np.nan
        r_c = float(task_metric_obj.r[idx_pos]) if idx_pos < len(task_metric_obj.r) else np.nan
        f1_c = float(task_metric_obj.f1[idx_pos]) if idx_pos < len(task_metric_obj.f1) else (
            (2 * p_c * r_c / (p_c + r_c)) if (p_c + r_c) > 0 else 0.0
        )
        ap50_c = float(task_metric_obj.ap50[idx_pos]) if idx_pos < len(task_metric_obj.ap50) else np.nan
        ap_c = float(task_metric_obj.ap[idx_pos]) if idx_pos < len(task_metric_obj.ap) else np.nan
        per_class_rows.append({
            "class_id": cid_int,
            "class_name": cname,
            "Precision": round(p_c, 4),
            "Recall": round(r_c, 4),
            "F1_score": round(f1_c, 4),
            "AP50": round(ap50_c, 4),
            "AP50_95": round(ap_c, 4),
        })

df_per_class = pd.DataFrame(per_class_rows)
print("\n=== MÉTRIQUES PAR CLASSE ===")
display(df_per_class)

# 5. Matrice de confusion sous forme de tableau
df_cm = None
if getattr(metrics, "confusion_matrix", None) is not None:
    cm_mat = metrics.confusion_matrix.matrix
    labels_cm = [model.names[i] for i in sorted(model.names.keys())] + ["background"]
    if cm_mat.shape[0] == len(labels_cm):
        df_cm = pd.DataFrame(cm_mat.astype(int), index=[f"Pred_{l}" for l in labels_cm], columns=[f"True_{l}" for l in labels_cm])
    else:
        df_cm = pd.DataFrame(cm_mat.astype(int))
    print("\n=== MATRICE DE CONFUSION ===")
    display(df_cm)

# 6. Métriques par image (si disponibles dans cette version d'Ultralytics)
df_per_image = None
img_metrics = getattr(metrics, "image_metrics", None)
if isinstance(img_metrics, dict) and len(img_metrics) > 0:
    df_per_image = pd.DataFrame.from_dict(img_metrics, orient="index").reset_index().rename(columns={"index": "image"})
    print("\n=== MÉTRIQUES PAR IMAGE (aperçu) ===")
    display(df_per_image.head(20))

=== MÉTRIQUES GLOBALES ===


,Metrique,Valeur
0,metrics/precision(B),0.954959
1,metrics/recall(B),0.356250
2,metrics/mAP50(B),0.531333
3,metrics/mAP50-95(B),0.398856
4,fitness,0.398856
5,BOX/Precision_moyenne (mp),0.954959
6,BOX/Recall_moyen (mr),0.356250
7,BOX/F1_moyen,0.518917
8,BOX/mAP50,0.531333
9,BOX/mAP75,0.499828



=== MÉTRIQUES PAR CLASSE ===


,class_id,class_name,Precision,Recall,F1_score,AP50,AP50_95
0,0,cable,1.0000,0.0000,0.0000,0.0553,0.0139
1,1,cable-tie,1.0000,0.0000,0.0000,0.7450,0.5478
2,2,clip,1.0000,0.0000,0.0000,0.0000,0.0000
3,3,connector,0.8675,0.7812,0.8221,0.8614,0.6297
4,4,label,0.9073,1.0000,0.9514,0.9950,0.8029



=== MATRICE DE CONFUSION ===


,True_cable,True_cable-tie,True_clip,True_connector,True_label,True_background
Pred_cable,0,0,0,0,0,1
Pred_cable-tie,0,0,0,0,0,0
Pred_clip,0,0,0,0,0,0
Pred_connector,1,0,0,57,0,14
Pred_label,0,0,0,0,4,0
Pred_background,30,2,5,7,0,0


## Bloc 4 — Visualisation des courbes de validation & Export des rapports

In [22]:
# 1. Export CSV et JSON complets
df_global.to_csv(save_dir / "metrics_globales.csv", index=False, encoding="utf-8-sig")
if not df_per_class.empty:
    df_per_class.to_csv(save_dir / "metrics_par_classe.csv", index=False, encoding="utf-8-sig")
if df_cm is not None:
    df_cm.to_csv(save_dir / "confusion_matrix.csv", encoding="utf-8-sig")
if df_per_image is not None:
    df_per_image.to_csv(save_dir / "metrics_par_image.csv", index=False, encoding="utf-8-sig")

# Export natif Ultralytics (CSV / JSON) si disponible
if hasattr(metrics, "to_csv"):
    (save_dir / "ultralytics_summary.csv").write_text(metrics.to_csv(), encoding="utf-8")
if hasattr(metrics, "to_json"):
    (save_dir / "ultralytics_summary.json").write_text(metrics.to_json(), encoding="utf-8")

print(f"Tous les tableaux CSV/JSON ont été exportés dans : {save_dir}")

# 2. Affichage des courbes et graphiques générés par Ultralytics
plot_files = sorted(save_dir.glob("*.png"))
if plot_files:
    n_plots = len(plot_files)
    cols = 2
    rows = (n_plots + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(16, 6 * rows))
    axes = np.array(axes).reshape(-1)
    for i, img_p in enumerate(plot_files):
        im = cv2.imread(str(img_p))
        axes[i].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        axes[i].set_title(img_p.name)
        axes[i].axis("off")
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Aucune image .png trouvée dans", save_dir)

Tous les tableaux CSV/JSON ont été exportés dans : D:\S5\Projet_d_expertise\projet\evaluations\03_orientation_des_composants_val


<Figure size 1600x1800 with 6 Axes>